# Exploratory Data Analysis on Personal Email Data

# 1. Environment setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn wordcloud pytz

In [ ]:
import os
import re
import csv
import mailbox

import numpy as np
import pandas as pd
import pytz

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('pandas version:', pd.__version__)

In [ ]:
import os, glob

MBOX_FOLDER = 'data'
MY_EMAIL    = 'your.email@gmail.com'
TIMEZONE    = 'Asia/Kolkata'
RAW_CSV     = 'mailbox.csv'

MBOX_FILES = sorted(
    f for f in glob.glob(os.path.join(MBOX_FOLDER, '*.mbox'))
    if os.path.getsize(f) > 0
)

print('Found', len(MBOX_FILES), 'usable mbox file(s):')
for f in MBOX_FILES:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)')

# 2. Data acquisition

## 2.1 Inspecting the raw mbox structure

In [ ]:
import mailbox

total = 0
for path in MBOX_FILES:
    mb = mailbox.mbox(path, create=False)
    n = len(mb)
    print(f'{os.path.basename(path)} -> {n:,} messages')
    total += n

print(f'\nTotal messages: {total:,}')

# inspect the headers of the first message
mb = mailbox.mbox(MBOX_FILES[0], create=False)
first = next(iter(mb))
print('\nHeaders in the first message:')
for key in first.keys():
    print(' ', key)

## 2.2 Extracting six fields to CSV

In [ ]:
import csv
from email.header import decode_header, make_header

def decode_field(value):
    # Subjects can arrive encoded like =?UTF-8?B?4KSo4KSu?=
    # This turns them back into readable text.
    if value is None:
        return None
    try:
        return str(make_header(decode_header(value)))
    except Exception:
        return str(value)


if not os.path.exists(RAW_CSV):
    count, skipped = 0, 0

    with open(RAW_CSV, 'w', newline='', encoding='utf-8') as outputfile:
        writer = csv.writer(outputfile)
        writer.writerow(['subject', 'from', 'to', 'date', 'label', 'thread'])

        for path in MBOX_FILES:
            print('Reading', os.path.basename(path))
            mb = mailbox.mbox(path, create=False)

            for message in mb:
                try:
                    writer.writerow([
                        decode_field(message['subject']),
                        decode_field(message['from']),
                        decode_field(message['to']),
                        message['date'],
                        message['X-Gmail-Labels'],
                        message['X-GM-THRID'],
                    ])
                    count += 1
                except Exception:
                    skipped += 1

                if count and count % 5000 == 0:
                    print(f'  ...{count:,} written')

    print(f'\nDone. Wrote {count:,} messages. Skipped {skipped} damaged.')
else:
    print('CSV already exists. Delete mailbox.csv if you want to rebuild it.')

In [ ]:
print('CSV size:', round(os.path.getsize(RAW_CSV)/1e6, 2), 'MB')

check = pd.read_csv(RAW_CSV)
print('Rows:', len(check))
print('\nColumn types:')
print(check.dtypes)
check.head()

# 3. Data cleansing

## 3.1 Loading the extracted data

In [ ]:
dfs = pd.read_csv(RAW_CSV)

print('Shape:', dfs.shape)
print('\nDtypes:')
print(dfs.dtypes)
dfs.head()

In [ ]:
dfs = pd.read_csv(RAW_CSV)
print('Shape:', dfs.shape)
print(dfs.dtypes)
dfs.head()

## 3.2 Date parsing - initial attempt fails on 99.32% of records

In [ ]:
before = len(dfs)

dfs['date'] = pd.to_datetime(dfs['date'], errors='coerce', utc=True)

bad = dfs['date'].isna().sum()
print(f'Rows before        : {before:,}')
print(f'Unparseable dates  : {bad:,}  ({100*bad/before:.2f}%)')

print('\nNew dtype of date:', dfs['date'].dtype)

## 3.3 Diagnosis: values that parse vs values that fail

In [ ]:
raw = pd.read_csv(RAW_CSV)

print('--- 10 raw date values ---')
for v in raw['date'].head(10):
    print(repr(v))

print('\n--- which ones fail? ---')
parsed = pd.to_datetime(raw['date'], errors='coerce', utc=True)
print('failed:', parsed.isna().sum(), 'of', len(raw))

print('\n--- 10 FAILING values ---')
for v in raw.loc[parsed.isna(), 'date'].head(10):
    print(repr(v))

print('\n--- 5 SUCCEEDING values ---')
for v in raw.loc[parsed.notna(), 'date'].head(5):
    print(repr(v))

## 3.4 Resolution - RFC 5322 parser recovers all 26,910 records

In [ ]:
from email.utils import parsedate_to_datetime

def parse_email_date(s):
    # RFC 5322 parser - handles GMT, +0000, +0530, (UTC) comments,
    # and single- or double-digit days.
    if not isinstance(s, str):
        return pd.NaT
    try:
        return parsedate_to_datetime(s)
    except Exception:
        return pd.NaT


dfs = pd.read_csv(RAW_CSV)
before = len(dfs)

dfs['date'] = dfs['date'].apply(parse_email_date)
dfs['date'] = pd.to_datetime(dfs['date'], errors='coerce', utc=True)

bad = dfs['date'].isna().sum()
print(f'Rows before       : {before:,}')
print(f'Unparseable dates : {bad:,}  ({100*bad/before:.2f}%)')
print(f'Successfully parsed: {before-bad:,}')
print('\nNew dtype:', dfs['date'].dtype)
print('Date range:', dfs["date"].min(), '->', dfs["date"].max())

## 3.5 Removing unusable rows

In [ ]:
dfs = dfs[dfs['date'].notna()].copy()

print(f'Rows retained: {len(dfs):,} of {before:,}')
print(f'Rows dropped : {before - len(dfs):,}')

## 3.6 Data quality audit

In [ ]:
quality = pd.DataFrame({
    'non_null' : dfs.notna().sum(),
    'null'     : dfs.isna().sum(),
    'null_pct' : (100 * dfs.isna().mean()).round(2),
    'unique'   : dfs.nunique(),
    'dtype'    : dfs.dtypes.astype(str),
})
quality

## 3.7 De-duplication

In [ ]:
dupes = dfs.duplicated(subset=['subject', 'from', 'date']).sum()
print(f'Duplicate messages found: {dupes:,}  ({100*dupes/len(dfs):.2f}%)')

dfs = dfs.drop_duplicates(subset=['subject', 'from', 'date']).copy()
print(f'Rows after de-duplication: {len(dfs):,}')

## 3.8 Checkpoint

In [ ]:
dfs.to_csv('gmail_clean.csv', index=False)
print('Saved:', 'gmail_clean.csv')
print('Final shape:', dfs.shape)

# 4. Feature engineering

## 4.1 Sender address normalisation (1,561 -> 984 senders)

In [ ]:
import re

def extract_email_id(string):
    # Pull the address out of an RFC 5322 From/To field.
    # Handles both "Name <a@b.com>" and bare "a@b.com".
    if not isinstance(string, str):
        return np.nan
    email = re.findall(r'<(.+?)>', string)
    if not email:
        email = list(filter(lambda y: '@' in y, string.split()))
    return email[0].strip().lower() if email else np.nan


print('BEFORE:')
print(dfs['from'].head(5).to_string(), '\n')

dfs['from'] = dfs['from'].apply(extract_email_id)
dfs['to']   = dfs['to'].apply(extract_email_id)

print('AFTER:')
print(dfs['from'].head(5).to_string())

print(f'\nUnique senders before cleaning: 1,561')
print(f'Unique senders after cleaning : {dfs["from"].nunique():,}')

## 4.2 Direction labelling - sent vs received

In [ ]:
dfs['label'] = np.where(dfs['from'] == MY_EMAIL.lower(), 'sent', 'inbox')

print(dfs['label'].value_counts())

n_sent = (dfs['label'] == 'sent').sum()
if n_sent == 0:
    print('\n[!] WARNING: no sent mail found. Check MY_EMAIL.')
    print('Top senders in your data:')
    print(dfs['from'].value_counts().head(10))
else:
    print(f'\nRatio: {(dfs["label"]=="inbox").sum()/n_sent:.1f} received per 1 sent')

## 4.3 Validating the label against Gmail's own Sent flag

In [ ]:
original = pd.read_csv('gmail_clean.csv')

print('--- Gmail\'s own labels (top 20) ---')
print(original['label'].value_counts().head(20))

print('\n--- messages Gmail marked as Sent ---')
gmail_sent = original['label'].fillna('').str.contains('Sent', case=False)
print('Count:', gmail_sent.sum())

print('\n--- our detection ---')
print('Count:', (dfs['label'] == 'sent').sum())

## 4.4 Recovering Gmail's category classifier and read state

In [ ]:
# Gmail's labels were overwritten, so recover them from the checkpoint.
original = pd.read_csv('gmail_clean.csv')
original['date'] = pd.to_datetime(original['date'], utc=True)

# align on the same keys we de-duplicated by
gmail_labels = (original
    .drop_duplicates(subset=['subject', 'from', 'date'])
    .set_index(original.drop_duplicates(subset=['subject','from','date']).index)['label'])

dfs['gmail_label'] = gmail_labels.reindex(dfs.index).fillna('')

# --- engagement: did I ever open it? ---
dfs['is_unread'] = dfs['gmail_label'].str.contains('Unread', case=False, na=False)

# --- Gmail's own category ---
def get_category(lbl):
    for cat in ['Promotions', 'Updates', 'Social', 'Personal',
                'Purchases', 'Bills', 'Travel', 'Forums']:
        if f'Category {cat}' in str(lbl):
            return cat
    return 'Uncategorised'

dfs['category'] = dfs['gmail_label'].apply(get_category)

# --- other flags ---
dfs['is_spam']      = dfs['gmail_label'].str.contains('Spam', case=False, na=False)
dfs['is_important'] = dfs['gmail_label'].str.contains('Important', case=False, na=False)

print('UNREAD RATE')
print(f'  Unread: {dfs["is_unread"].sum():,} ({100*dfs["is_unread"].mean():.1f}%)')
print(f'  Opened: {(~dfs["is_unread"]).sum():,}')

print('\nCATEGORY BREAKDOWN')
print(dfs['category'].value_counts())

print(f'\nSpam: {dfs["is_spam"].sum():,}  |  Important: {dfs["is_important"].sum():,}')

## 4.5 Engagement by category

In [ ]:
engagement = (dfs[dfs['label'] == 'inbox']
              .groupby('category')
              .agg(total=('is_unread', 'size'),
                   unread=('is_unread', 'sum'))
              .assign(unread_pct=lambda d: (100 * d['unread'] / d['total']).round(1))
              .sort_values('total', ascending=False))

print(engagement)

## 4.6 Temporal features

In [ ]:
import pytz

local_tz = pytz.timezone(TIMEZONE)

dfs['date'] = dfs['date'].apply(lambda x: x.astimezone(local_tz))

# NOTE: the book uses x.weekday_name — removed in pandas 1.0.
# day_name() is the current API.
dfs['dayofweek'] = dfs['date'].apply(lambda x: x.day_name())
dfs['dayofweek'] = pd.Categorical(
    dfs['dayofweek'],
    categories=['Monday','Tuesday','Wednesday','Thursday',
                'Friday','Saturday','Sunday'],
    ordered=True)

dfs['timeofday'] = dfs['date'].apply(lambda x: x.hour + x.minute/60 + x.second/3600)
dfs['hour']      = dfs['date'].apply(lambda x: x.hour)
dfs['year_int']  = dfs['date'].apply(lambda x: x.year)
dfs['year']      = dfs['date'].apply(lambda x: x.year + x.dayofyear/365.25)
dfs['month']     = dfs['date'].apply(lambda x: x.month)
dfs['is_weekend'] = dfs['dayofweek'].isin(['Saturday','Sunday'])

dfs.index = dfs['date']
dfs = dfs.drop(columns=['date']).sort_index()

print('Time features created.\n')
print(dfs[['dayofweek','hour','year_int','is_weekend']].head())
print('\nRange:', dfs.index.min().strftime('%d %b %Y'),
      '->', dfs.index.max().strftime('%d %b %Y'))

# 5. Descriptive statistics

## 5.1 Dataset summary

In [ ]:
span_start, span_end = dfs.index.min(), dfs.index.max()
span_days = (span_end - span_start).days

received = dfs[dfs['label'] == 'inbox']
sent     = dfs[dfs['label'] == 'sent']

print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'From              : {span_start:%a, %d %b %Y}')
print(f'To                : {span_end:%a, %d %b %Y}')
print(f'Span              : {span_days:,} days ({span_days/365.25:.1f} years)')
print(f'Total messages    : {len(dfs):,}')
print(f'  Received        : {len(received):,}')
print(f'  Sent            : {len(sent):,}')
print(f'Unique senders    : {dfs["from"].nunique():,}')
print(f'Unique threads    : {dfs["thread"].nunique():,}')
print(f'Unread rate       : {100*dfs["is_unread"].mean():.1f}%')
print(f'Mean msgs/day     : {len(dfs)/span_days:.2f}')
print('=' * 60)

## 5.2 Distribution of daily volume

In [ ]:
daily = dfs.resample('D').size()

print('DAILY EMAIL VOLUME')
print(f'  Mean          : {daily.mean():.2f}')
print(f'  Median        : {daily.median():.2f}')
print(f'  Mode          : {daily.mode().iloc[0]}')
print(f'  Std deviation : {daily.std():.2f}')
print(f'  Variance      : {daily.var():.2f}')
print(f'  Skewness      : {daily.skew():.3f}')
print(f'  Kurtosis      : {daily.kurtosis():.3f}')
print(f'  Min / Max     : {daily.min()} / {daily.max()}')
print(f'  Q1 / Q3       : {daily.quantile(.25):.1f} / {daily.quantile(.75):.1f}')
print(f'  Busiest day   : {daily.idxmax():%d %b %Y} ({daily.max()} emails)')
print(f'  Zero-mail days: {(daily == 0).sum():,}')

# 6. Analysis

## 6.1 Bivariate and multivariate analysis: day of week x hour of day

In [ ]:
pivot = (dfs.groupby(['dayofweek', 'hour'], observed=False)
            .size()
            .unstack(fill_value=0)
            .reindex(columns=range(24), fill_value=0))

plt.figure(figsize=(14, 4))
sns.heatmap(pivot, cmap='YlGnBu', linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Number of emails'})
plt.title('Email activity: day of week × hour of day', fontsize=13, pad=12)
plt.xlabel('Hour of day (IST)')
plt.ylabel('')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

peak = pivot.stack()
pd_, ph = peak.idxmax()
print(f'Peak cell: {pd_} at {ph:02d}:00 — {peak.max():,} emails')
print(f'Quietest hour overall: {pivot.sum().idxmin():02d}:00')

night = pivot.loc[:, [0,1,2,3,4]].values.sum()
print(f'Emails 00:00–05:00: {night:,} ({100*night/len(dfs):.1f}%)')

## 6.2 Long-run volume trend

In [ ]:
monthly = dfs.resample('MS').size()
monthly_full = monthly.iloc[1:-1]   # drop partial Jan 2022 and Aug 2026

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.fill_between(monthly_full.index, monthly_full.values, alpha=0.25, color='#3E8E7E')
ax.plot(monthly_full.index, monthly_full.values, lw=1, alpha=0.7, color='#3E8E7E',
        label='Monthly volume')
ax.plot(monthly_full.index, monthly_full.rolling(6, min_periods=1).mean(),
        lw=2.5, color='#1F3B57', label='6-month rolling mean')
ax.set_ylabel('Emails per month')
ax.set_title('Email volume over time (Feb 2022 – Jul 2026)\nPartial first/last months excluded')
ax.legend()
plt.tight_layout(); plt.savefig('trend.png', dpi=200, bbox_inches='tight'); plt.show()

## 6.3 Significance testing of the trend

In [ ]:
from scipy import stats
import numpy as np

m = monthly_full.copy()
x = np.arange(len(m))
slope, intercept, r, p, se = stats.linregress(x, m.values)

print(f'n = {len(m)} months')
print(f'Trend: {slope:+.1f} emails per month')
print(f'R² = {r**2:.3f}, p = {p:.2e}')
print('Significant' if p < 0.05 else 'Not significant')

## 6.4 Extreme value investigation

In [ ]:
busiest = dfs.loc['2025-02-28']
print(f'Emails that day: {len(busiest)}\n')
print('By category:')
print(busiest['category'].value_counts())
print('\nTop senders:')
print(busiest['from'].value_counts().head(10))

## 6.5 Sender concentration

In [ ]:
sender_counts = received['from'].value_counts()
cum = sender_counts.cumsum() / sender_counts.sum()

print(f'Total senders: {len(sender_counts):,}\n')
for n in [1, 5, 10, 25, 50, 100]:
    if len(cum) >= n:
        print(f'Top {n:>3} senders = {cum.iloc[n-1]*100:5.1f}% of inbox')

print(f'\nSenders who emailed only once: {(sender_counts == 1).sum():,}')

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sender_counts.head(15).iloc[::-1].plot(kind='barh', ax=ax[0], color='#2E5EAA')
ax[0].set_title('Top 15 senders'); ax[0].set_xlabel('Emails')

ax[1].plot(np.arange(1, len(cum)+1), cum.values*100, color='#3E8E7E', lw=2)
ax[1].set_xscale('log')
ax[1].set_xlabel('Number of senders (log scale)')
ax[1].set_ylabel('Cumulative % of inbox')
ax[1].set_title('Sender concentration')
plt.tight_layout(); plt.savefig('senders.png', dpi=200, bbox_inches='tight'); plt.show()
print(f'\nSenders who emailed exactly once: {(sender_counts==1).sum():,}')
print(f'They contribute: {100*(sender_counts==1).sum()/len(received):.1f}% of inbox')
print(f'\nTop sender alone: {sender_counts.iloc[0]:,} emails '
      f'({100*sender_counts.iloc[0]/len(received):.1f}%)')

In [ ]:
inbox = dfs[dfs['label'] == 'inbox']

eng = (inbox[inbox['category'] != 'Uncategorised']
       .groupby('category')
       .agg(total=('is_unread','size'), unread=('is_unread','sum')))
eng['unread_pct'] = 100 * eng['unread'] / eng['total']
eng = eng[eng['total'] >= 25].sort_values('total', ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

eng['total'].plot(kind='bar', ax=ax[0], color='#2E5EAA', rot=45)
ax[0].set_title('Volume received by category')
ax[0].set_ylabel('Emails'); ax[0].set_xlabel('')

bars = ax[1].bar(eng.index, eng['unread_pct'], color='#E07A2F')
ax[1].axhline(92.7, ls='--', color='k', lw=1, label='Overall 92.7%')
ax[1].set_ylim(0, 105); ax[1].set_ylabel('% never opened')
ax[1].set_title('Engagement: unread rate by category')
ax[1].tick_params(axis='x', rotation=45); ax[1].legend()
for b, v in zip(bars, eng['unread_pct']):
    ax[1].text(b.get_x()+b.get_width()/2, v+1, f'{v:.0f}%', ha='center', fontsize=9)

plt.tight_layout(); plt.savefig('engagement.png', dpi=200, bbox_inches='tight'); plt.show()
print(eng.round(1))